In [1]:
# notebook to compile all of the csvs into a single array
import os
import yaml
import numpy as np

import cantera as ct


# check the dimensions of the species/reaction/base delay files against the sensitivity points in the sim_config matrix

In [2]:
working_dir = '/scratch/harris.se/guassian_scratch/test_sens_20260313'
mech_yaml = os.path.join(working_dir, 'chem_annotated.yaml')

gas = ct.Solution(mech_yaml)

# check the size
conditions_dict_path = os.path.join(working_dir, 'sim_config.yaml')
# conditions_dict_path = os.path.join(os.environ['AUTOSCIENCE_REPO'], 'experiment', 'butane1.yaml')
if not os.path.exists(conditions_dict_path):
    logging.warning(f'Expected to find sim_config.yaml at {conditions_dict_path} but it does not exist. Please copy it to the directory with your mech file.')
    raise FileNotFoundError(f'sim_config.yaml not found at {conditions_dict_path}')

with open(conditions_dict_path) as f:
    conditions_dict = yaml.safe_load(f)

base_delays = np.load(os.path.join(working_dir, 'sensitivity', 'base_delays.npy'))
sample_spec_delays = np.load(os.path.join(working_dir, 'sensitivity', 'spec_delay_0000.npy'))
sample_reaction_delays = np.load(os.path.join(working_dir, 'sensitivity', 'reaction_delay_000000.npy'))


K = len(conditions_dict['sensitivity_points'])
assert len(base_delays) == K
assert len(sample_spec_delays) == K
assert len(sample_reaction_delays) == K




In [3]:
# Build big table of sensitivity delays
perturbed_delays = np.zeros((gas.n_species + gas.n_reactions, K))

for i in range(gas.n_species):
    spec_file = os.path.join(working_dir, 'sensitivity', f'spec_delay_{i:04}.npy')
    if not os.path.exists(spec_file):
        print(f'missing species {i:04}')
        continue
    perturbed_delays[i, :] = np.load(spec_file)

for i in range(gas.n_reactions):
    rxn_file = os.path.join(working_dir, 'sensitivity', f'reaction_delay_{i:06}.npy')
    if not os.path.exists(rxn_file):
        print(f'missing reaction {i:06}')
        continue
    perturbed_delays[gas.n_species + i, :] = np.load(rxn_file)


In [4]:
# save the resulting delay array
np.save(os.path.join(working_dir, 'total_perturbed_mech_delays.npy'), perturbed_delays)